# Preprocessing All Three Files

In [12]:
import os

# Set of valid opcodes (you can expand this as needed)
valid_opcodes_set = {
    'mov', 'add', 'sub', 'cmp', 'jmp', 'call', 'ret', 'lea', 'push', 'pop',
    'inc', 'dec', 'or', 'and', 'xor', 'not', 'test', 'shl', 'shr', 'jz',
    'jnz', 'jb', 'ja', 'jno', 'je', 'jne', 'jbe', 'jae', 'movzx', 'movsx', 
    'imul', 'idiv', 'div', 'retn', 'db', 'dw', 'equ'
}

# Function to read opcode sequences from a list of .asm files
def load_and_preprocess_asm_files(file_paths):
    all_opcodes = []
    for file_path in file_paths:
        with open(file_path, 'r', encoding='latin-1') as file:
            lines = file.readlines()

        opcodes = []
        for line in lines:
            line = line.strip()
            if not line or line.startswith(';'):  # Ignore comments and empty lines
                continue
            
            tokens = line.split()
            if tokens:  # Ensure there are tokens in the line
                opcode = tokens[0]

                # Ignore labels and non-opcode keywords
                if opcode.lower() in valid_opcodes_set:
                    opcodes.append(opcode)  # Extract valid opcodes
        
        all_opcodes.append(opcodes)
    return all_opcodes

# Example usage: paths to your .asm files
file_paths = ['IDAN1.asm', 'IDAN2.asm', 'IDAN3.asm']
opcode_sequences = load_and_preprocess_asm_files(file_paths)

# Check if sequences are loaded
for i, seq in enumerate(opcode_sequences, 1):
    print(f"First 10 opcodes from IDAN{i}.asm: {seq[:10]}")


First 10 opcodes from IDAN1.asm: ['call', 'jno', 'test', 'sub', 'add', 'or', 'add', 'push', 'pop', 'mov']
First 10 opcodes from IDAN2.asm: ['call', 'pop', 'sub', 'jmp', 'mov', 'and', 'mov', 'mov', 'mov', 'div']
First 10 opcodes from IDAN3.asm: ['call', 'mov', 'add', 'mov', 'sub', 'jmp', 'db', 'db', 'call', 'mov']


# Training the HMM with All Sequences

In [13]:
from hmmlearn import hmm
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Prepare the opcode sequences for HMM training
def prepare_sequences(opcode_sequences):
    label_encoder = LabelEncoder()
    encoded_sequences = [label_encoder.fit_transform(seq) for seq in opcode_sequences]
    return encoded_sequences, label_encoder

# Fit Label Encoder and encode the sequences
encoded_sequences, label_encoder = prepare_sequences(opcode_sequences)

# Flatten the sequences for HMM training
X = np.concatenate(encoded_sequences).reshape(-1, 1)
lengths = [len(seq) for seq in encoded_sequences]

# Train an HMM
model = hmm.MultinomialHMM(n_components=2, n_iter=100)
model.fit(X, lengths)

# Save the trained model and label encoder
import pickle
with open('hmm_model.pkl', 'wb') as model_file, open('label_encoder.pkl', 'wb') as encoder_file:
    pickle.dump(model, model_file)
    pickle.dump(label_encoder, encoder_file)


MultinomialHMM has undergone major changes. The previous version was implementing a CategoricalHMM (a special case of MultinomialHMM). This new implementation follows the standard definition for a Multinomial distribution (e.g. as in https://en.wikipedia.org/wiki/Multinomial_distribution). See these issues for details:
https://github.com/hmmlearn/hmmlearn/issues/335
https://github.com/hmmlearn/hmmlearn/issues/340


In [18]:
import pickle
from hmmlearn import hmm
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Load the saved model and label encoder
with open('hmm_model.pkl', 'rb') as model_file, open('label_encoder.pkl', 'rb') as encoder_file:
    model = pickle.load(model_file)
    label_encoder = pickle.load(encoder_file)

# Extend the label encoder with all possible opcodes
all_possible_opcodes = [
    'mov', 'add', 'sub', 'cmp', 'jmp', 'call', 'ret', 'lea', 'push', 'pop',
    'inc', 'dec', 'or', 'and', 'xor', 'not', 'test', 'shl', 'shr', 'jz',
    'jnz', 'jb', 'ja', 'jno', 'je', 'jne', 'jbe', 'jae', 'movzx', 'movsx', 
    'imul', 'idiv', 'div', 'retn', 'db', 'dw', 'equ'
]

# Refit the label encoder with all possible opcodes
label_encoder.fit(all_possible_opcodes)

# Function to read opcode sequences from an ASM file
def load_and_preprocess_asm_file(file_path):
    with open(file_path, 'r', encoding='latin-1') as file:
        lines = file.readlines()
    
    opcodes = []
    for line in lines:
        line = line.strip()
        if not line or line.startswith(';'):  # Ignore comments and empty lines
            continue
        
        tokens = line.split()
        if tokens:  # Ensure there are tokens in the line
            opcode = tokens[0]
            if opcode.lower() in all_possible_opcodes:
                opcodes.append(opcode)  # Extract valid opcodes
    
    return opcodes

# Function to classify opcode sequences
def classify_opcode_sequence(opcode_sequence, model, label_encoder, threshold=-50):
    encoded_sequence = label_encoder.transform(opcode_sequence).reshape(-1, 1)
    log_likelihood = model.score(encoded_sequence)
    
    # Classify based on log-likelihood
    if log_likelihood < threshold:
        return "Malware"
    else:
        return "Legit"

# Paths to the ASM files
legit_file_path = 'test1.asm'
malware_file_path = 'test2.asm'

# Load and preprocess the ASM files
legit_opcode_sequence = load_and_preprocess_asm_file(legit_file_path)
malware_opcode_sequence = load_and_preprocess_asm_file(malware_file_path)

# Classify the opcode sequences
prediction_legit = classify_opcode_sequence(legit_opcode_sequence, model, label_encoder)
prediction_malware = classify_opcode_sequence(malware_opcode_sequence, model, label_encoder)

print(f"Prediction for legit sequence: {prediction_legit}")
print(f"Prediction for malware sequence: {prediction_malware}")


Prediction for legit sequence: Legit
Prediction for malware sequence: Legit


In [34]:
import numpy as np
import os
from sklearn.model_selection import train_test_split
from hmmlearn import hmm
import joblib

def read_opcode_files(file_names, encoding='ISO-8859-1'):
    sequences = []
    for file_name in file_names:
        with open(file_name, 'r', encoding=encoding) as file:
            for line in file:
                line = line.strip()
                # Skip empty lines or lines that are comments
                if not line or line.startswith(';') or 'segment' in line or 'assume' in line:
                    continue
                # Split the line into opcodes (this assumes that opcodes are separated by whitespace)
                opcodes = line.split()
                # Append only if the line contains opcodes
                if opcodes:  
                    sequences.append(opcodes)
                    print(sequences[:10])
    return sequences

# Function to convert opcodes to numerical features
def opcodes_to_features(sequences):
    unique_opcodes = set(op for seq in sequences for op in seq)
    opcode_to_idx = {opcode: idx for idx, opcode in enumerate(unique_opcodes)}
    
    feature_matrix = []
    for seq in sequences:
        feature_matrix.append([opcode_to_idx[opcode] for opcode in seq if opcode in opcode_to_idx])  # Ensure valid opcodes
    
    return feature_matrix, opcode_to_idx

# Function to train HMM model
def train_hmm_model(sequences, n_components):
    lengths = [len(seq) for seq in sequences]
    X = np.concatenate(sequences).reshape(-1, 1)

    model = hmm.MultinomialHMM(n_components=n_components, n_iter=100)
    model.fit(X, lengths)
    
    return model

# Function to classify unseen opcode sequences
def classify_sequence(model, sequence):
    sequence_idx = np.array(sequence).reshape(-1, 1)
    log_prob = model.score(sequence_idx)
    return log_prob  # Return log probability for better handling of small values

def main():
    file_names = ['IDAN1.asm', 'IDAN2.asm', 'IDAN3.asm']
    sequences = read_opcode_files(file_names)
    for i, sequence in enumerate(opcode_sequences[:10]):
       
          print(f"Sequence {i + 1}: {sequence}")
    feature_matrix, opcode_to_idx = opcodes_to_features(sequences)
    
    X_train, X_test = train_test_split(feature_matrix, test_size=0.2, random_state=42)
    
    hmm_model = train_hmm_model(X_train, n_components=5)
    
    joblib.dump(hmm_model, 'hmm_model.pkl')
    
    # Example of classifying an unseen opcode sequence (malware)
    malware_sequence = ['xor', 'add', 'jmp']  # Replace with actual opcodes
    malware_features = [opcode_to_idx.get(opcode) for opcode in malware_sequence if opcode in opcode_to_idx]

    if malware_features:
        malware_log_prob = classify_sequence(hmm_model, malware_features)
        print(f"Malware Sequence Log Probability: {malware_log_prob}")
        
        # Classify based on threshold (adjustable)
        threshold = -10.0  # Adjust based on your validation data
        if malware_log_prob > threshold:
            print("The malware sequence is classified as malware.")
        else:
            print("The malware sequence is classified as legitimate.")
    else:
        print("Malware sequence contains invalid entries.")
    
    # Example of classifying a non-malware opcode sequence
    non_malware_sequence = ['mov', 'add', 'sub', 'jmp', 'cmp', 'je', 'mov', 'push', 'pop', 'call', 'ret']
    non_malware_features = [opcode_to_idx.get(opcode) for opcode in non_malware_sequence if opcode in opcode_to_idx]

    if non_malware_features:
        non_malware_log_prob = classify_sequence(hmm_model, non_malware_features)
        print(f"Non-Malware Sequence Log Probability: {non_malware_log_prob}")
        
        # Classify based on threshold (adjustable)
        if non_malware_log_prob > threshold:
            print("The non-malware sequence is classified as malware.")
        else:
            print("The non-malware sequence is classified as legitimate.")
    else:
        print("Non-malware sequence contains invalid entries.")

if __name__ == "__main__":
    main()


MultinomialHMM has undergone major changes. The previous version was implementing a CategoricalHMM (a special case of MultinomialHMM). This new implementation follows the standard definition for a Multinomial distribution (e.g. as in https://en.wikipedia.org/wiki/Multinomial_distribution). See these issues for details:
https://github.com/hmmlearn/hmmlearn/issues/335
https://github.com/hmmlearn/hmmlearn/issues/340
IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



Malware Sequence Log Probability: 5.551115123125783e-17
The malware sequence is classified as malware.
Non-Malware Sequence Log Probability: 0.0
The non-malware sequence is classified as malware.
